# 10 Canopy Baseline


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

if Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

ART = ROOT / 'artifacts'
MODEL_DIR = ART / 'models'
for p in [ART, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)


In [ ]:
from glob import glob
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

feature_paths = sorted(glob(str(ROOT / 'artifacts' / 'features' / 'features_*_10m.parquet')))
if not feature_paths and Path('/kaggle/input').exists():
    feature_paths = sorted(str(p) for p in Path('/kaggle/input').rglob('features_*_10m.parquet'))

if not feature_paths:
    raise FileNotFoundError('No feature parquet found locally or in /kaggle/input')

features = pd.concat([pd.read_parquet(p) for p in feature_paths], ignore_index=True)
features = features.dropna(subset=['target_chm']).copy()

exclude = {'tile_id','zone_epsg','grid_x','grid_y','target_chm','target_landcover'}
cols = [c for c in features.columns if c not in exclude and pd.api.types.is_numeric_dtype(features[c])]
X = features[cols].fillna(0)
y = features['target_chm']

model = DummyRegressor(strategy='mean')
model.fit(X,y)
pred = model.predict(X)
metrics = {
    'mae': float(mean_absolute_error(y,pred)),
    'rmse': float(np.sqrt(mean_squared_error(y,pred))),
    'r2': float(r2_score(y,pred)),
}

joblib.dump({'model':model,'feature_columns':cols}, MODEL_DIR/'canopy_baseline.joblib')
row = pd.DataFrame([{'project':'canopy','model_name':'baseline',**metrics}])
out_csv = MODEL_DIR / 'model_results.csv'
if out_csv.exists():
    old = pd.read_csv(out_csv)
    row = pd.concat([old,row], ignore_index=True)
row.to_csv(out_csv, index=False)
metrics


In [ ]:
import pandas as pd
pd.read_csv(MODEL_DIR/'model_results.csv').tail(10)
